# MammoRisk+ Module 3: build the CBIS-DDSM PNG cache on Kaggle (O2/O3)

Streams CBIS-DDSM from the public TCIA API and writes the 1024×512 16-bit PNG cache plus lesion masks.
**The 164 GB of DICOM is never stored**: each series is downloaded, converted and deleted
(`module3_classifier/fetch_cbis.py`).

**Settings before running:** Notebook options → **Internet: On**. No GPU needed (choose CPU).
Use **Save Version → Save & Run All** so it runs in the background (Kaggle limit: 12 h).
Afterwards: Output → **New Dataset** (keep it **private**) → name it `mammorisk-cbis-png-1024x512`.
The training notebook then adds that dataset as input.

CBIS-DDSM licence: CC BY 3.0 (TCIA). Cite Lee et al. 2017 (Sci Data 4:170177) + TCIA.

In [ ]:
!git clone --depth 1 https://github.com/FaiqaZarar/MammoRisk-Plus.git /kaggle/working/MammoRisk-Plus-code
%cd /kaggle/working/MammoRisk-Plus-code
!git log --oneline -1
!pip -q install pydicom

## Fetch
- The patient-level split (`splits/m3_cbis_split.csv`, 3,032 images / 1,566 patients) is in the repo,
  so the split is fixed **before** any image is seen.
- Resumable: if the run stops, run the same cell again and only the missing or failed images are fetched.
- `--workers 8`: parallel downloads. Please don't go higher (TCIA is a shared public service).

In [ ]:
CACHE = "/kaggle/working/cbis_png_1024x512"
!mkdir -p /kaggle/tmp
!python -W ignore -m module3_classifier.fetch_cbis --config configs/m3_effnetb3.yaml     --cache-dir {CACHE} --tmp-dir /kaggle/tmp --workers 8

In [ ]:
# Retry pass for anything that failed (network hiccups)
!python -W ignore -m module3_classifier.fetch_cbis --config configs/m3_effnetb3.yaml     --cache-dir {CACHE} --tmp-dir /kaggle/tmp --workers 4

## Check the cache

In [ ]:
import pandas as pd, numpy as np, cv2, matplotlib.pyplot as plt
man = pd.read_csv(f"{CACHE}/manifest.csv")
print(man.status.value_counts())
print(man.groupby(["split", "status"]).size().unstack(fill_value=0))
mal = man[(man.label == 1) & (man.status == "ok")]
print("malignant images with a mask:", mal.mask_path.notna().sum(), "/", len(mal))
print("lesions whose mask was not found:", man.masks_not_found.fillna("").str.len().gt(0).sum())
!du -sh {CACHE}

In [ ]:
# Spot-check 8 random malignant images: the breast should face LEFT and the red outline should sit on the lesion
s = mal[mal.mask_path.notna()].sample(8, random_state=0)
fig, ax = plt.subplots(1, 8, figsize=(24, 6))
for a, r in zip(ax, s.itertuples()):
    im = cv2.imread(f"{CACHE}/{r.cache_path}", cv2.IMREAD_UNCHANGED) / 65535
    m = cv2.imread(f"{CACHE}/{r.mask_path}", 0) > 0
    a.imshow(im, cmap="gray"); a.contour(m, colors="r", linewidths=1); a.set_title(r.image_id, fontsize=8); a.axis("off")
plt.show()

## Clean up
Only the cache folder should be in the output (the code clone is removed so the dataset contains
just images, masks and the manifest).

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/MammoRisk-Plus-code /kaggle/tmp
!ls /kaggle/working